In [1]:
import pandas as pd
import pypsa
import gurobipy

YEAR = 2030
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_{YEAR}.csv"
costs = pd.read_csv(url, index_col=[0, 1])
costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
costs = costs.value.unstack().fillna({"discount rate": 0.07, "lifetime": 20, "FOM": 0})

In [2]:
costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]
def annuity(r: float, n: int) -> float:
    return r / (1.0 - 1.0 / (1.0 + r) ** n)
a = costs.apply(lambda x: annuity(x["discount rate"], x["lifetime"]), axis=1)
costs["capital_cost"] = (a + costs["FOM"] / 100) * costs["investment"]

In [3]:
costs["marginal_cost"] 

technology
Alkaline electrolyzer large size    NaN
Alkaline electrolyzer medium size   NaN
Alkaline electrolyzer small size    NaN
Ammonia cracker                     NaN
BEV Bus city                        NaN
                                     ..
uranium                             NaN
waste CHP                           NaN
waste CHP CC                        NaN
water tank charger                  NaN
water tank discharger               NaN
Name: marginal_cost, Length: 294, dtype: float64

In [4]:
RESOLUTION = 3  # hours
url = "https://tubcloud.tu-berlin.de/s/9toBssWEdaLgHzq/download/time-series.csv"
ts = pd.read_csv(url, index_col=0, parse_dates=True)[::RESOLUTION]

In [5]:
n = pypsa.Network()
n.add("Bus", "electricity", carrier="electricity")
n.set_snapshots(ts.index)
n.snapshot_weightings.loc[:, :] = RESOLUTION

In [6]:
carriers = [
    "wind",
    "solar",
    "hydrogen storage",
    "battery storage",
    "load shedding",
    "electrolysis",
    "turbine",
    "electricity",
    "hydrogen",
]
colors = [
    "dodgerblue",
    "gold",
    "black",
    "yellowgreen",
    "darkorange",
    "magenta",
    "red",
    "grey",
    "grey",
]
n.add("Carrier", carriers, color=colors)

Index(['wind', 'solar', 'hydrogen storage', 'battery storage', 'load shedding',
       'electrolysis', 'turbine', 'electricity', 'hydrogen'],
      dtype='object')

In [7]:
n.add(
    "Load",
    "demand",
    bus="electricity",
    p_set=ts.load_mw,
)

Index(['demand'], dtype='object')

In [8]:
n.add(
    "Generator",
    "load shedding",
    bus="electricity",
    carrier="load shedding",
    marginal_cost=2000,
    p_nom=ts.load_mw.max(),
)
n.add(
    "Generator",
    "wind",
    bus="electricity",
    carrier="wind",
    p_max_pu=ts.wind_pu,
    capital_cost=costs.at["onwind", "capital_cost"],
    marginal_cost=costs.at["onwind", "marginal_cost"],
    p_nom_extendable=True,
)
n.add(
    "Generator",
    "solar",
    bus="electricity",
    carrier="solar",
    p_max_pu=ts.pv_pu,
    capital_cost=costs.at["solar", "capital_cost"],
    marginal_cost=costs.at["solar", "marginal_cost"],
    p_nom_extendable=True,
)
n.add(
    "StorageUnit",
    "battery storage",
    bus="electricity",
    carrier="battery storage",
    max_hours=3,
    capital_cost=costs.at["battery inverter", "capital_cost"]
    + 3 * costs.at["battery storage", "capital_cost"],
    efficiency_store=costs.at["battery inverter", "efficiency"],
    efficiency_dispatch=costs.at["battery inverter", "efficiency"],
    p_nom_extendable=True,
    cyclic_state_of_charge=True,
)

Index(['battery storage'], dtype='object')

In [9]:
n.optimize(solver_name="highs")

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 6/6 [00:00<00:00, 251.66it/s]
INFO:linopy.io: Writing time: 0.18s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 17523 primals, 40883 duals
Objective: 9.84e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [16]:
snaps = n.snapshots[:100]

In [19]:
n.optimize.create_model()
m = n.model

In [28]:
n.snapshot_weightings.iloc[2:,2] = 5

In [30]:
(m['Generator-p'] * n.snapshot_weightings.loc[:,"generators"]).sum()

LinearExpression
----------------
+3 Generator-p[2019-01-01 00:00:00, load shedding] + 3 Generator-p[2019-01-01 00:00:00, wind] + 3 Generator-p[2019-01-01 00:00:00, solar] ... +5 Generator-p[2019-12-31 21:00:00, load shedding] + 5 Generator-p[2019-12-31 21:00:00, wind] + 5 Generator-p[2019-12-31 21:00:00, solar]

In [17]:
n.snapshot_weightings.loc[snaps, "generators"]

snapshot
2019-01-01 00:00:00    3.0
2019-01-01 03:00:00    3.0
2019-01-01 06:00:00    3.0
2019-01-01 09:00:00    3.0
2019-01-01 12:00:00    3.0
                      ... 
2019-01-12 21:00:00    3.0
2019-01-13 00:00:00    3.0
2019-01-13 03:00:00    3.0
2019-01-13 06:00:00    3.0
2019-01-13 09:00:00    3.0
Name: generators, Length: 100, dtype: float64

In [10]:
n.statistics.optimal_capacity().div(1e3)

component    carrier        
Generator    load shedding      10.901160
             solar              43.728701
             wind               38.960252
StorageUnit  battery storage    28.451648
dtype: float64

In [10]:
n.statistics.energy_balance.iplot()

In [11]:
n.optimize.create_model()
m = n.model


In [18]:
n.snapshots

DatetimeIndex(['2019-01-01 00:00:00', '2019-01-01 03:00:00',
               '2019-01-01 06:00:00', '2019-01-01 09:00:00',
               '2019-01-01 12:00:00', '2019-01-01 15:00:00',
               '2019-01-01 18:00:00', '2019-01-01 21:00:00',
               '2019-01-02 00:00:00', '2019-01-02 03:00:00',
               ...
               '2019-12-30 18:00:00', '2019-12-30 21:00:00',
               '2019-12-31 00:00:00', '2019-12-31 03:00:00',
               '2019-12-31 06:00:00', '2019-12-31 09:00:00',
               '2019-12-31 12:00:00', '2019-12-31 15:00:00',
               '2019-12-31 18:00:00', '2019-12-31 21:00:00'],
              dtype='datetime64[ns]', name='snapshot', length=2920, freq=None)

In [17]:
costs.loc["onwind","marginal_cost"]

np.float64(nan)

In [12]:
m.objective

Objective:
----------
LinearExpression: +6000 Generator-p[2019-01-01 00:00:00, load shedding] + 6000 Generator-p[2019-01-01 03:00:00, load shedding] + 6000 Generator-p[2019-01-01 06:00:00, load shedding] ... +1.016e+05 Generator-p_nom[wind] + 5.135e+04 Generator-p_nom[solar] + 6.367e+04 StorageUnit-p_nom[battery storage]
Sense: min
Value: None

In [24]:
m.to_netcdf("../networks/test.nc") 

In [ ]:
m.add_variables()